In [36]:
# HW0: robust mega acquisition (Colab-first, git-safe)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/egil10/stk-mat2011.git"
REPO_DIR = Path("/content/stk-mat2011") if IN_COLAB else Path.cwd().resolve().parents[1]

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # keep notebook reproducible if you re-run later
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

    # optional but useful when Colab VM is fresh
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

# make scripts importable
SCRIPTS_DIR = REPO_DIR / "code" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from p_duka import download_and_save

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")

IN_COLAB=True
REPO_DIR=/content/stk-mat2011
SCRIPTS_DIR=/content/stk-mat2011/code/scripts


In [37]:
from datetime import datetime
import shutil
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

# Repo output path (where p_duka writes)
repo_processed = REPO_DIR / "code" / "data" / "processed"
repo_processed.mkdir(parents=True, exist_ok=True)

# Persistent cache on Drive (survives git pull/reclone)
drive_processed = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed") if IN_COLAB else repo_processed
if IN_COLAB:
    drive_processed.mkdir(parents=True, exist_ok=True)


def month_list(start_ym: str, end_ym: str):
    """Inclusive YYYYMM range."""
    y0, m0 = int(start_ym[:4]), int(start_ym[4:6])
    y1, m1 = int(end_ym[:4]), int(end_ym[4:6])
    out = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        out.append(f"{y}{m:02d}")
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return out


def expected_files(pair: str, yyyymm: str):
    s = pair.replace("/", "").lower()
    return [
        f"{s}_dukascopy_bid_{yyyymm}.parquet",
        f"{s}_dukascopy_ask_{yyyymm}.parquet",
    ]


def sync_drive_to_repo():
    if not IN_COLAB:
        return
    copied = 0
    for fp in drive_processed.glob("*.parquet"):
        tgt = repo_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced Drive -> repo: {copied} new file(s)")


def sync_repo_to_drive():
    if not IN_COLAB:
        return
    copied = 0
    for fp in repo_processed.glob("*.parquet"):
        tgt = drive_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced repo -> Drive: {copied} new file(s)")


print(f"repo_processed={repo_processed}")
print(f"drive_processed={drive_processed}")

Mounted at /content/drive
repo_processed=/content/stk-mat2011/code/data/processed
drive_processed=/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed


In [38]:
# Fast acquisition: newest-first + chunked + periodic sync
PAIR = "EUR/USD"

# 1) Choose one chunk per run (recommended)
# Example quick chunk:
START_YYYYMM = "201901"
END_YYYYMM   = "201912"
SYNC_EVERY = 1
MAX_FAIL_BEFORE_STOP = 3

# Pull persistent files into repo first
sync_drive_to_repo()

months = month_list(START_YYYYMM, END_YYYYMM)
months = list(reversed(months))  # newest first
print(f"Planned months: {len(months)} ({months[-1]} -> {months[0]}) newest-first")

ok, skip, fail = [], [], []
ok_since_sync = 0

for i, yyyymm in enumerate(months, 1):
    names = expected_files(PAIR, yyyymm)
    have = all((drive_processed / n).exists() or (repo_processed / n).exists() for n in names)

    if have:
        skip.append(yyyymm)
        if i % 10 == 0:
            print(f"[{i}/{len(months)}] skip {yyyymm} (already exists)")
        continue

    print(f"[{i}/{len(months)}] fetch {PAIR} {yyyymm}")
    try:
        _ = download_and_save(PAIR, yyyymm, compression="snappy")
        ok.append(yyyymm)
        ok_since_sync += 1

        # periodic sync to persistent storage
        if ok_since_sync >= SYNC_EVERY:
            sync_repo_to_drive()
            ok_since_sync = 0

    except Exception as e:
        print(f"  ERROR {yyyymm}: {e}")
        fail.append(yyyymm)

        # safety stop if endpoint is flaky
        if len(fail) >= MAX_FAIL_BEFORE_STOP:
            print(f"Stopping early: reached {MAX_FAIL_BEFORE_STOP} failures.")
            break

# final sync
sync_repo_to_drive()

print("\nDone")
print(f"Downloaded: {len(ok)}")
print(f"Skipped existing: {len(skip)}")
print(f"Failed: {len(fail)}")
if ok:
    print("Newest downloaded months:", ok[:10])   # because newest-first
if fail:
    print("Failed months:", fail[:20])

Synced Drive -> repo: 0 new file(s)
Planned months: 12 (201901 -> 201912) newest-first
[1/12] fetch EUR/USD 201912


INFO:DUKASCRIPT:current timestamp :2019-12-02T11:52:30.715000
INFO:DUKASCRIPT:current timestamp :2019-12-02T16:32:31.324000
INFO:DUKASCRIPT:current timestamp :2019-12-03T07:12:16.151000
INFO:DUKASCRIPT:current timestamp :2019-12-03T13:16:02.054000
INFO:DUKASCRIPT:current timestamp :2019-12-03T19:38:21.479000
INFO:DUKASCRIPT:current timestamp :2019-12-04T09:39:05.562000
INFO:DUKASCRIPT:current timestamp :2019-12-04T15:01:31.873000
INFO:DUKASCRIPT:current timestamp :2019-12-04T23:28:41.460000
INFO:DUKASCRIPT:current timestamp :2019-12-05T12:02:21.459000
INFO:DUKASCRIPT:current timestamp :2019-12-05T18:29:16.538000
INFO:DUKASCRIPT:current timestamp :2019-12-06T09:46:24.539000
INFO:DUKASCRIPT:current timestamp :2019-12-06T15:17:28.989000
INFO:DUKASCRIPT:current timestamp :2019-12-09T06:08:49.112000
INFO:DUKASCRIPT:current timestamp :2019-12-09T15:29:20.029000
INFO:DUKASCRIPT:current timestamp :2019-12-10T07:49:59.105000
INFO:DUKASCRIPT:current timestamp :2019-12-10T14:48:25.857000
INFO:DUK

  Received 1,622,152 ticks
Synced repo -> Drive: 2 new file(s)
[2/12] fetch EUR/USD 201911


INFO:DUKASCRIPT:current timestamp :2019-11-01T11:15:44.614000
INFO:DUKASCRIPT:current timestamp :2019-11-01T15:05:23.034000
INFO:DUKASCRIPT:current timestamp :2019-11-04T06:48:13.192000
INFO:DUKASCRIPT:current timestamp :2019-11-04T13:05:48.112000
INFO:DUKASCRIPT:current timestamp :2019-11-04T21:37:54.109000
INFO:DUKASCRIPT:current timestamp :2019-11-05T09:03:22.334000
INFO:DUKASCRIPT:current timestamp :2019-11-05T14:17:16.266000
INFO:DUKASCRIPT:current timestamp :2019-11-05T19:32:22.747000
INFO:DUKASCRIPT:current timestamp :2019-11-06T08:47:44.073000
INFO:DUKASCRIPT:current timestamp :2019-11-06T15:37:17.571000
INFO:DUKASCRIPT:current timestamp :2019-11-07T03:13:06.067000
INFO:DUKASCRIPT:current timestamp :2019-11-07T10:34:45.037000
INFO:DUKASCRIPT:current timestamp :2019-11-07T15:04:35.570000
INFO:DUKASCRIPT:current timestamp :2019-11-07T20:51:22.717000
INFO:DUKASCRIPT:current timestamp :2019-11-08T10:03:36.757000
INFO:DUKASCRIPT:current timestamp :2019-11-08T15:32:38.665000
INFO:DUK

  Received 1,536,373 ticks
Synced repo -> Drive: 2 new file(s)
[3/12] fetch EUR/USD 201910


...

  Received 2,755,889 ticks
Synced repo -> Drive: 2 new file(s)
[11/12] fetch EUR/USD 201902


INFO:DUKASCRIPT:current timestamp :2019-02-01T08:02:03.118000
INFO:DUKASCRIPT:current timestamp :2019-02-01T11:16:23.316000
INFO:DUKASCRIPT:current timestamp :2019-02-01T14:09:47.276000
INFO:DUKASCRIPT:current timestamp :2019-02-01T16:24:23.233000
INFO:DUKASCRIPT:current timestamp :2019-02-01T21:42:59.608000
INFO:DUKASCRIPT:current timestamp :2019-02-04T07:36:56.696000
INFO:DUKASCRIPT:current timestamp :2019-02-04T10:46:07.476000
INFO:DUKASCRIPT:current timestamp :2019-02-04T14:39:10.040000
INFO:DUKASCRIPT:current timestamp :2019-02-04T18:00:01.109000
INFO:DUKASCRIPT:current timestamp :2019-02-05T03:34:45.301000
INFO:DUKASCRIPT:current timestamp :2019-02-05T09:33:17.032000
INFO:DUKASCRIPT:current timestamp :2019-02-05T13:38:46.040000
INFO:DUKASCRIPT:current timestamp :2019-02-05T16:32:43.865000
INFO:DUKASCRIPT:current timestamp :2019-02-05T23:28:19.836000
INFO:DUKASCRIPT:current timestamp :2019-02-06T07:55:20.189000
INFO:DUKASCRIPT:current timestamp :2019-02-06T11:40:17.538000
INFO:DUK

  Received 2,740,372 ticks
Synced repo -> Drive: 2 new file(s)
[12/12] fetch EUR/USD 201901


INFO:DUKASCRIPT:current timestamp :2019-01-02T06:11:01.068000
INFO:DUKASCRIPT:current timestamp :2019-01-02T09:14:43.432000
INFO:DUKASCRIPT:current timestamp :2019-01-02T12:26:27.361000
INFO:DUKASCRIPT:current timestamp :2019-01-02T14:46:26.459000
INFO:DUKASCRIPT:current timestamp :2019-01-02T17:27:57.017000
INFO:DUKASCRIPT:current timestamp :2019-01-02T21:57:56.826000
INFO:DUKASCRIPT:current timestamp :2019-01-03T02:05:36.068000
INFO:DUKASCRIPT:current timestamp :2019-01-03T07:06:52.842000
INFO:DUKASCRIPT:current timestamp :2019-01-03T09:48:07.125000
INFO:DUKASCRIPT:current timestamp :2019-01-03T13:15:09.926000
INFO:DUKASCRIPT:current timestamp :2019-01-03T15:25:21.940000
INFO:DUKASCRIPT:current timestamp :2019-01-03T17:12:04.385000
INFO:DUKASCRIPT:current timestamp :2019-01-03T20:35:27.797000
INFO:DUKASCRIPT:current timestamp :2019-01-04T02:52:50.595000
INFO:DUKASCRIPT:current timestamp :2019-01-04T08:43:49.116000
INFO:DUKASCRIPT:current timestamp :2019-01-04T12:34:39.701000
INFO:DUK

  Received 3,477,166 ticks
Synced repo -> Drive: 2 new file(s)
Synced repo -> Drive: 0 new file(s)

Done
Downloaded: 12
Skipped existing: 0
Failed: 0
Newest downloaded months: ['201912', '201911', '201910', '201909', '201908', '201907', '201906', '201905', '201904', '201903']


In [39]:
# Manifest / health check
store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))
print(f"Store: {store}")
print(f"Total parquet files: {len(files)}")

total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
print(f"Total size: {total_mb:.1f} MB")

rows = []
for fp in files:
    name = fp.name
    # expected format: eurusd_dukascopy_bid_202101.parquet
    parts = name.replace(".parquet", "").split("_")
    if len(parts) >= 4:
        yyyymm = parts[-1]
        side = parts[-2]
    else:
        yyyymm = ""
        side = ""
    rows.append({"file": name, "yyyymm": yyyymm, "side": side, "mb": fp.stat().st_size / 1e6})

m = pd.DataFrame(rows)
if len(m):
    print("\nFiles by month (head):")
    display(m.sort_values(["yyyymm", "side"]).head(20))

    month_counts = m.groupby("yyyymm").size().rename("n_files").reset_index().sort_values("yyyymm")
    print("\nMonth coverage (last 24):")
    display(month_counts.tail(24))

# Note for gitignore/git pull workflow:
# - Keep parquet in Drive path above (not in git).
# - Notebook syncs between repo output and Drive cache.
# - After git pull/reclone, data remains in Drive and is re-synced.

Store: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Total parquet files: 176
Total size: 3461.5 MB

Files by month (head):


,file,yyyymm,side,mb
0,eurusd_dukascopy_ask_201901.parquet,201901,ask,32.903542
88,eurusd_dukascopy_bid_201901.parquet,201901,bid,33.013018
1,eurusd_dukascopy_ask_201902.parquet,201902,ask,25.910599
89,eurusd_dukascopy_bid_201902.parquet,201902,bid,25.923784
2,eurusd_dukascopy_ask_201903.parquet,201903,ask,26.050396
90,eurusd_dukascopy_bid_201903.parquet,201903,bid,26.065307
3,eurusd_dukascopy_ask_201904.parquet,201904,ask,25.506756
91,eurusd_dukascopy_bid_201904.parquet,201904,bid,25.523916
4,eurusd_dukascopy_ask_201905.parquet,201905,ask,23.970251
92,eurusd_dukascopy_bid_201905.parquet,201905,bid,23.970107



Month coverage (last 24):


,yyyymm,n_files
64,202405,2
65,202406,2
66,202407,2
67,202408,2
68,202409,2
69,202410,2
70,202411,2
71,202412,2
72,202501,2
73,202502,2


In [40]:
# End-of-HW0 summary: Drive processed folder + month coverage
import re
from pathlib import Path

import pandas as pd

store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))

# Filename pattern: eurusd_dukascopy_bid_202101.parquet
pat = re.compile(
    r"^(?P<sym>.+)_dukascopy_(?P<side>bid|ask)_(?P<ym>\d{6})\.parquet$",
    re.I,
)

rows = []
for fp in files:
    m = pat.match(fp.name)
    if not m:
        rows.append(
            {
                "file": fp.name,
                "symbol": "",
                "side": "",
                "yyyymm": "",
                "rows": None,
                "mb": fp.stat().st_size / 1e6,
            }
        )
        continue
    try:
        import pyarrow.parquet as pq
        nr = pq.ParquetFile(fp).metadata.num_rows
    except Exception:
        nr = len(pd.read_parquet(fp, columns=["datetime"]))

    rows.append(
        {
            "file": fp.name,
            "symbol": m.group("sym").upper(),
            "side": m.group("side").lower(),
            "yyyymm": m.group("ym"),
            "rows": int(nr),
            "mb": fp.stat().st_size / 1e6,
        }
    )

meta = pd.DataFrame(rows)
parsed = meta[meta["yyyymm"] != ""].copy()

total_bytes = sum(f.stat().st_size for f in files)
n_files = len(files)

print("=" * 60)
print("HW0 — processed store summary")
print("=" * 60)
print(f"Folder: {store}")
print(f"Parquet files: {n_files}")
print(f"Total size: {total_bytes / (1024**3):.3f} GB  ({total_bytes / (1024**2):.1f} MB)")

if len(parsed):
    bid = parsed[parsed["side"] == "bid"].rename(columns={"rows": "rows_bid", "mb": "mb_bid"})
    ask = parsed[parsed["side"] == "ask"].rename(columns={"rows": "rows_ask", "mb": "mb_ask"})
    cov = (
        bid[["symbol", "yyyymm", "rows_bid", "mb_bid", "file"]]
        .merge(
            ask[["symbol", "yyyymm", "rows_ask", "mb_ask", "file"]],
            on=["symbol", "yyyymm"],
            how="outer",
            suffixes=("_bidfile", "_askfile"),
        )
        .sort_values(["symbol", "yyyymm"])
    )
    cov["has_bid"] = cov["rows_bid"].notna()
    cov["has_ask"] = cov["rows_ask"].notna()
    cov["complete_pair"] = cov["has_bid"] & cov["has_ask"]
    cov["year"] = cov["yyyymm"].str[:4]

    rows_bid = int(cov["rows_bid"].fillna(0).sum())
    rows_ask = int(cov["rows_ask"].fillna(0).sum())

    print()
    print(f"Rows bid side (sum of bid files): {rows_bid:,}")
    print(f"Rows ask side (sum of ask files): {rows_ask:,}")
    print(f"Rows all files summed:            {rows_bid + rows_ask:,}")
    print()
    print(
        f"Months with complete bid+ask: {int(cov['complete_pair'].sum())} "
        f"/ {len(cov)} rows in coverage table"
    )
    print(f"Months missing bid: {int((~cov['has_bid']).sum())}")
    print(f"Months missing ask: {int((~cov['has_ask']).sum())}")

    print("\n--- By year (complete bid+ask months) ---")
    byy = (
        cov.groupby("year")
        .agg(
            months_total=("yyyymm", "nunique"),
            months_complete=("complete_pair", "sum"),
            rows_bid=("rows_bid", "sum"),
            rows_ask=("rows_ask", "sum"),
        )
        .sort_index()
    )
    display(byy)

    print("\n--- Month coverage (symbol, yyyymm) — tail ---")
    display(
        cov[
            [
                "symbol",
                "yyyymm",
                "has_bid",
                "has_ask",
                "complete_pair",
                "rows_bid",
                "rows_ask",
            ]
        ].tail(36)
    )

    # Optional: list gaps inside [min_ym, max_ym] for each symbol
    print("\n--- Missing YYYYMM inside observed range (per symbol) ---")
    for sym, g in cov.groupby("symbol"):
        yms = sorted(g["yyyymm"].unique())
        if not yms:
            continue
        lo, hi = yms[0], yms[-1]

        def nxt(ym):
            y, m = int(ym[:4]), int(ym[4:6])
            if m == 12:
                return f"{y+1}01"
            return f"{y}{m+1:02d}"

        have = set(yms)
        missing = []
        cur = lo
        while cur <= hi:
            if cur not in have:
                missing.append(cur)
            if cur == hi:
                break
            cur = nxt(cur)
        print(f"{sym}: range {lo}..{hi} | missing inside range: {len(missing)}")
        if missing:
            print("  ", missing[:40], ("..." if len(missing) > 40 else ""))

else:
    print("\n(No Dukascopy-style filenames parsed — check naming.)")

print()
print("Approx. unique tick times: use bid row sum when bid/ask are paired.")
print("=" * 60)

HW0 — processed store summary
Folder: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Parquet files: 176
Total size: 3.380 GB  (3461.5 MB)

Rows bid side (sum of bid files): 193,078,365
Rows ask side (sum of ask files): 193,078,365
Rows all files summed:            386,156,730

Months with complete bid+ask: 88 / 88 rows in coverage table
Months missing bid: 0
Months missing ask: 0

--- By year (complete bid+ask months) ---


,months_total,months_complete,rows_bid,rows_ask
year,,,,
2019,12,12,29190340,29190340
2020,12,12,32763984,32763984
2021,12,12,16797709,16797709
2022,12,12,36950672,36950672
2023,12,12,27553506,27553506
2024,12,12,20690554,20690554
2025,12,12,23463391,23463391
2026,4,4,5668209,5668209



--- Month coverage (symbol, yyyymm) — tail ---


,symbol,yyyymm,has_bid,has_ask,complete_pair,rows_bid,rows_ask
52,EURUSD,202305,True,True,True,2236715,2236715
53,EURUSD,202306,True,True,True,2051718,2051718
54,EURUSD,202307,True,True,True,2114645,2114645
55,EURUSD,202308,True,True,True,2238800,2238800
56,EURUSD,202309,True,True,True,1737379,1737379
57,EURUSD,202310,True,True,True,2199274,2199274
58,EURUSD,202311,True,True,True,1938713,1938713
59,EURUSD,202312,True,True,True,2113635,2113635
60,EURUSD,202401,True,True,True,2173014,2173014
61,EURUSD,202402,True,True,True,1709509,1709509



--- Missing YYYYMM inside observed range (per symbol) ---
EURUSD: range 201901..202604 | missing inside range: 0

Approx. unique tick times: use bid row sum when bid/ask are paired.
